<a href="https://colab.research.google.com/github/rohitmadde0213-hub/rfm_framework/blob/main/pyspark1_sql.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("SQL Queries with PySpark") \
    .getOrCreate()


In [ ]:
# Load a CSV file into a DataFrame
df = spark.read.option('header', 'true').csv('sales.csv')

In [ ]:
# Register the DataFrame as a SQL temporary view

df.createOrReplaceTempView("sales")

In [ ]:
bills = spark.sql("""with
bills as
(
  SELECT InvoiceNo,
  (CAST(Quantity AS DECIMAL(10, 2)) * CAST(UnitPrice AS DECIMAL(10, 2))) AS AMOUNT
  FROM sales
)
select
InvoiceNo, round(sum(AMOUNT),2) as total, count(*) counts
from bills
group by InvoiceNo""")
bills.createOrReplaceTempView("bills")
bills.show()

In [ ]:
result_sql1 = spark.sql("""
select *
from sales
""")
result_sql1.show()

In [ ]:
rfm = spark.sql("""
    with cte as (
        select
            s.CustomerID,
            to_date(max(s.InvoiceDate)) as last_purchase_date,
            to_date(min(s.InvoiceDate)) as first_purchase_date,
            count(distinct s.InvoiceNo) as num_purchase,
            round(sum(b.total), 2) as monetary
        from sales s
        left join bills b
            on s.InvoiceNo = b.InvoiceNo
        group by s.CustomerID
    )
    select
        CustomerID,
        last_purchase_date,
        first_purchase_date,
        num_purchase,
        monetary,
        reference_date,
        datediff(reference_date, last_purchase_date) as recency,
        round(num_purchase / month_cust, 2) as frequency
    from (
        select
            *,
            date_add(max(last_purchase_date) over (), 1) as reference_date,
            greatest(cast(months_between(last_purchase_date, first_purchase_date) as int), 0) + 1 as month_cust
        from cte
    )
""")

In [ ]:
rfm.show()

convert into Quantile

In [ ]:
rfm.createOrReplaceTempView("rfm")
percentiles_df = spark.sql("""
    SELECT approx_percentile(monetary, array(0.0, 0.2, 0.4, 0.6, 0.8, 1.0)) AS percentiles
    FROM rfm
""")

percentiles_df.show(truncate=False)

In [ ]:
rfm_quantile = spark.sql("""
    SELECT
        a.*,
        -- for monetary
        b.percentile[1] as m20,
        b.percentile[2] as m40,
        b.percentile[3] as m60,
        b.percentile[4] as m80,
        b.percentile[5] as m100,
        -- for frequency
        c.percentile[1] as f20,
        c.percentile[2] as f40,
        c.percentile[3] as f60,
        c.percentile[4] as f80,
        c.percentile[5] as f100,
        -- for recency
        d.percentile[1] as d20,
        d.percentile[2] as d40,
        d.percentile[3] as d60,
        d.percentile[4] as d80,
        d.percentile[5] as d100
    FROM rfm a
    CROSS JOIN (SELECT approx_percentile(monetary, array(0.0, 0.2, 0.4, 0.6, 0.8, 1.0)) AS percentile FROM rfm) b
    CROSS JOIN (SELECT approx_percentile(frequency, array(0.0, 0.2, 0.4, 0.6, 0.8, 1.0)) AS percentile FROM rfm) c
    CROSS JOIN (SELECT approx_percentile(recency, array(0.0, 0.2, 0.4, 0.6, 0.8, 1.0)) AS percentile FROM rfm) d
""")

rfm_quantile.createOrReplaceTempView("rfm_quantile")

In [ ]:
rfm_quantile.show()

Giving them score now based on the quantile

In [ ]:
rfm_scored_based_on_quantile = spark.sql("""
    SELECT
        CustomerID,
        m_score,
        f_score,
        r_score,
        monetary,
        frequency,
        recency,
        CAST(ROUND((f_score * m_score) / 2.0) AS BIGINT) AS fm_score
    FROM (
        SELECT
            *,
            CASE
                WHEN monetary <= m20 THEN 1
                WHEN monetary <= m40 AND monetary >= m20 THEN 2
                WHEN monetary <= m60 AND monetary >= m40 THEN 3
                WHEN monetary <= m80 AND monetary >= m60 THEN 4
                WHEN monetary <= m100 AND monetary >= m80 THEN 5
            END AS m_score,
            CASE
                WHEN frequency <= f20 THEN 1
                WHEN frequency <= f40 AND frequency >= f20 THEN 2
                WHEN frequency <= f60 AND frequency >= f40 THEN 3
                WHEN frequency <= f80 AND frequency >= f60 THEN 4
                WHEN frequency <= f100 AND frequency >= f80 THEN 5
            END AS f_score,
            CASE
                WHEN recency <= d20 THEN 5
                WHEN recency <= d40 AND recency >= d20 THEN 4
                WHEN recency <= d60 AND recency >= d40 THEN 3
                WHEN recency <= d80 AND recency >= d60 THEN 2
                WHEN recency <= d100 AND recency >= d80 THEN 1
            END AS r_score
        FROM rfm_quantile
    )
""")

rfm_scored_based_on_quantile.createOrReplaceTempView("rfm_final")

In [ ]:
rfm_scored_based_on_quantile.show()

Now doing the customer segmentation

In [ ]:
rfm_segmented = spark.sql("""
    SELECT
        CustomerID,
        recency, frequency, monetary,
        r_score, f_score, m_score,
        fm_score,
        CASE
            WHEN (r_score = 5 AND fm_score = 5)
              OR (r_score = 5 AND fm_score = 4)
              OR (r_score = 4 AND fm_score = 5)
            THEN 'Champions'
            WHEN (r_score = 5 AND fm_score = 3)
              OR (r_score = 4 AND fm_score = 4)
              OR (r_score = 3 AND fm_score = 5)
              OR (r_score = 3 AND fm_score = 4)
            THEN 'Loyal Customers'
            WHEN (r_score = 5 AND fm_score = 2)
              OR (r_score = 4 AND fm_score = 2)
              OR (r_score = 3 AND fm_score = 3)
              OR (r_score = 4 AND fm_score = 3)
            THEN 'Potential Loyalists'
            WHEN r_score = 5 AND fm_score = 1 THEN 'Recent Customers'
            WHEN (r_score = 4 AND fm_score = 1)
              OR (r_score = 3 AND fm_score = 1)
            THEN 'Promising'
            WHEN (r_score = 3 AND fm_score = 2)
              OR (r_score = 2 AND fm_score = 3)
              OR (r_score = 2 AND fm_score = 2)
            THEN 'Customers Needing Attention'
            WHEN r_score = 2 AND fm_score = 1 THEN 'About to Sleep'
            WHEN (r_score = 2 AND fm_score = 5)
              OR (r_score = 2 AND fm_score = 4)
              OR (r_score = 1 AND fm_score = 3)
            THEN 'At Risk'
            WHEN (r_score = 1 AND fm_score = 5)
              OR (r_score = 1 AND fm_score = 4)
            THEN 'Cant Lose Them'
            WHEN r_score = 1 AND fm_score = 2 THEN 'Hibernating'
            WHEN r_score = 1 AND fm_score = 1 THEN 'Lost'
        END AS rfm_segment
    FROM rfm_final
""")

# Register it for future queries or export
rfm_segmented.createOrReplaceTempView("rfm_segments")

rfm_segmented.show()

In [ ]:
# Preview the distribution of your customer segments
spark.sql("""
    SELECT rfm_segment, COUNT(CustomerID) AS customer_count, ROUND(AVG(monetary), 2) AS avg_monetary
    FROM rfm_segments
    GROUP BY rfm_segment
    ORDER BY customer_count DESC
""").show(truncate=False)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
numeric_cols = ['recency', 'frequency', 'monetary', 'r_score', 'f_score', 'm_score', 'fm_score']
pandas_df = rfm_segmented.select(numeric_cols).toPandas()

heatmap = sns.heatmap(pandas_df.corr(), annot=True, cmap='coolwarm')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# 1. Fetch data from Spark and convert to Pandas (filtering out NULLs for clean plotting)
rfm_pd_df = spark.sql("""
    SELECT rfm_segment, COUNT(CustomerID) AS customer_count, ROUND(AVG(monetary), 2) AS avg_monetary
    FROM rfm_segments
    WHERE rfm_segment IS NOT NULL
    GROUP BY rfm_segment
    ORDER BY customer_count DESC
""").toPandas()

# Convert 'avg_monetary' column to float to avoid TypeError with matplotlib.colors.Normalize
rfm_pd_df['avg_monetary'] = rfm_pd_df['avg_monetary'].astype(float)

# 2. Set up the Seaborn theme and figure size
plt.figure(figsize=(12, 7))
sns.set_theme(style="whitegrid")

# 3. Create a color mapping based on 'avg_monetary' using a Reds colormap
norm = plt.Normalize(rfm_pd_df['avg_monetary'].min(), rfm_pd_df['avg_monetary'].max())
colors = [plt.cm.Reds(norm(val)) for val in rfm_pd_df['avg_monetary']]

# 4. Draw the horizontal bar plot
ax = sns.barplot(
    data=rfm_pd_df,
    y='rfm_segment',
    x='customer_count',
    hue='rfm_segment', # Assign y variable to hue
    palette=colors,
    legend=False # Hide the legend as colors are represented by a colorbar
)

# 5. Add titles and axis labels
plt.title('RFM Customer Segments: Volume & Average Spend Analysis', fontsize=16, fontweight='bold', pad=15)
plt.xlabel('Customer Count (Volume)', fontsize=12, fontweight='bold')
plt.ylabel('RFM Segment', fontsize=12, fontweight='bold')

# 6. Add a colorbar legend for the Monetary value
sm = plt.cm.ScalarMappable(cmap="Reds", norm=norm)
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, orientation='vertical', pad=0.03)
cbar.set_label('Average Monetary Value ($)', rotation=270, labelpad=20, fontsize=12)

# 7. Add exact customer count labels next to each bar
for p in ax.patches:
    width = p.get_width()
    if width > 0:
        ax.annotate(f'{int(width)}',
                    (width + 1, p.get_y() + p.get_height() / 2.),
                    ha='left', va='center',
                    fontsize=10, color='black', fontweight='bold')

# Display the plot
plt.tight_layout()
plt.show()